## Tools

Allows AI to call functions/code to...
* extend knowledge - web search, database search
* carry out actions in an application - email, slack, calendars
* generate and execute code
* transform data and other calculations

In practice, code talks to llm about problem. LLM responds with tools it should use then code executes the functions

In Agentic AI, tools can be used to cool other LLMs (orchestration) or to track a "ToDo" list and progress towards a goal (Agentic Loop)


In [13]:
import os
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv
import json

load_dotenv(override=True)

True

In [2]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if not openai_api_key:
    print("OPENAI API Key not found")
elif not openai_api_key.startswith("sk-proj"):
    print("Invalid OpenAI API Key")
else:
    print("Valid OpenAI API Key")

if not google_api_key:
    print("Google API Key not found")
elif not google_api_key.startswith("AI"):
    print("Invlaid Google API Key")
else:
    print("Valid Google API Key")


Valid OpenAI API Key
Valid Google API Key


In [3]:
GPT_MODEL = 'gpt-4o-mini'
GEMINI_MODEL = 'gemini-2.5-flash'

openai = OpenAI()

google_url = gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=google_url, api_key=google_api_key)

In [66]:
system_message = """  
You are a helpful assistant for an Airline called Jet2Vacations.
Give short, courteous answers - no more than 2 sentences.
Always be accurate, if you don't know the answer then say so.

You have the tools to retrive prices from a SQL Lite data base or set prices in the database when given a city and ticket price
"""

In [5]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role":"user", "content": message}]
    response = openai.chat.completions.create(model=GPT_MODEL, messages = messages)
    return response.choices[0].message.content

In [7]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## Tools

In [8]:
ticket_prices = {'london': '£150', 'barcelona': '£50', 'new york': '£1150', 'munich': '£250'}

def get_ticket_price(destination):
    print(f"Tool called for {destination}")
    price = ticket_prices.get(destination.lower(), "Unknown City")
    return f"the price of a ticket to {destination} is {price}"

In [9]:
get_ticket_price('new york')

Tool called for new york


'the price of a ticket to new york is £1150'

In [19]:
#lets define our tool using json structure

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a ticket to destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ['destination'],
        "additionalProperties": False
    }
}




In [21]:
tools = [{"type":"function", "function": price_function}]

In [22]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a ticket to destination city',
   'parameters': {'type': 'object',
    'properties': {'destination': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination'],
    'additionalProperties': False}}}]

In [23]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [24]:

#if statement to handle tool calls
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination')
        price_details = get_ticket_price(city)
        response = {
            "role":"tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

# combine into chat function
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role":"user", "content": message}]
    response = openai.chat.completions.create(model=GPT_MODEL, messages = messages, tools=tools) #parse tools json
    
    if response.choices[0].finish_reason =="tool_calls": #finish reason determines what we do next
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=GPT_MODEL, messages=messages)

    return response.choices[0].message.content


In [25]:
gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Tool called for London


## Multiple tool calls

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name =="get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id" :tool_call.id
            })
    return responses

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role":"user", "content": message}]
    response = openai.chat.completions.create(model=GPT_MODEL, messages = messages, tools=tools) #parse tools json
    
    if response.choices[0].finish_reason =="tool_calls": #finish reason determines what we do next
        message = response.choices[0].message
        response = handle_tool_calls(message) #add in new tool call london
        messages.append(message) #append only adds a single item
        messages.extend(response) #extend not append - adds items iteratively
        response = openai.chat.completions.create(model=GPT_MODEL, messages=messages)

    return response.choices[0].message.content


In [31]:
gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


Tool called for Munich
Tool called for London


## Add a database to make it more realistic

In [79]:
#can use postgres lookup or api like superbase
import sqlite3

In [80]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (destination TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [81]:
#new tool function:

def get_ticket_price(destination):
    print(f"DATABASE TOOL CALLED: Getting price for {destination}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE destination = ?', (destination.lower(),))
        result = cursor.fetchone()
        return f" Ticket price to {destination} is £{result[0]}" if result else "No price data available for this destination"

In [82]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this destination'

In [86]:
# insert data into database
ticket_prices = {'london': 150, 'barcelona': 50, 'new york': 1150, 'munich': 250, 'tokyo':1500}

def set_ticket_price(destination, price):
    print(f"DATABASE TOOL CALLED: Setting price for {destination}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (destination, price) VALUES (?,?) ON CONFLICT(destination) DO UPDATE SET price = ?', (destination.lower(), price, price))
        conn.commit()

for destination, price in ticket_prices.items():
    set_ticket_price(destination, price)

DATABASE TOOL CALLED: Setting price for london
DATABASE TOOL CALLED: Setting price for barcelona
DATABASE TOOL CALLED: Setting price for new york
DATABASE TOOL CALLED: Setting price for munich
DATABASE TOOL CALLED: Setting price for tokyo


In [87]:
set_ticket_price("Beijing", 2500)

DATABASE TOOL CALLED: Setting price for Beijing


In [88]:
get_ticket_price("Beijing")

DATABASE TOOL CALLED: Getting price for Beijing


' Ticket price to Beijing is £2500.0'

In [39]:
# we have changed the get ticket price function thats called by chat


gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Tokyo


## Exercise

Add a tool to set the price of a ticket!

In [89]:
#json

set_price_function = {
    "name": "set_ticket_price",
    "description": "Use this when the user wants to set, update, or change the ticket price for a city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that a customer will want to travel to",
            },
            "price": {
                "type": "number",
                "description": "the price of the ticket to travel to the city"
            }
        },
        "required": ['destination', 'price'],
        "additionalProperties": False
    }
}

In [90]:
tools = [{"type":"function", "function": price_function},
         {"type": "function", "function": set_price_function}]

In [91]:



tool_registry ={ 
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price
}

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        function = tool_registry.get(tool_call.function.name)
        arguments = json.loads(tool_call.function.arguments)
        
        result = function(**arguments)

        responses.append({
            "role": "tool",
            "content": str(result),
            "tool_call_id": tool_call.id,
        })
    return responses

In [92]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role":"user", "content": message}]
    response = openai.chat.completions.create(model = GPT_MODEL, messages = messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)
        response = openai.chat.completions.create(model=GPT_MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type='messages').launch()


* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Setting price for Baku
DATABASE TOOL CALLED: Getting price for Munich
DATABASE TOOL CALLED: Getting price for Baku
